# Segmentação exploratória de clientes

Demonstração com dados inteiramente fictícios. Recupera o fluxo StandardScaler → KMeans do estudo original e acrescenta contratos, avaliação e exportação. Não reproduz a população original nem comprova retenção, conversão ou retorno financeiro.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/marketing').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from marketing.synthetic import generate
from marketing.contracts import validate
from marketing.model import analyze
from marketing.pipeline import publish


## Dados e contrato
Uma linha por cliente fictício. ID textual não participa do modelo. Idade em anos, renda anual em unidades monetárias fictícias e pontuação artificial de 0 a 100. Essa pontuação não mede frequência, fidelidade nem gasto monetário observado. Três componentes foram desenhados no gerador; resultados não validam segmentos comerciais.

In [ ]:
data = validate(generate(seed=42, n=180))
assert data.id_cliente.is_unique
data.describe()


## Comparação de k
StandardScaler ajustado às três variáveis. Candidatos de 2 a 6 quando viáveis; pelo menos cinco clientes por grupo. Seleção exploratória pela maior silhouette entre candidatos elegíveis; empate favorece menor k. Sementes 42, 73 e 101, com 20 inicializações. ARI mede concordância entre partições sem depender dos números dos rótulos. Não há variável alvo nem teste de acurácia.

In [ ]:
analysis = analyze(data)
analysis.metrics


## Perfis em unidades originais
Rótulos de cluster são arbitrários. Comparar idade, renda e pontuação; campanhas são hipóteses a testar, e não conclusões sobre recorrência ou fidelização.

In [ ]:
assert analysis.profiles.clientes.sum() == len(data)
analysis.profiles


## Extremos e sensibilidade
Diagnóstico IQR sem exclusão ou limitação silenciosa. Comparação StandardScaler versus RobustScaler com o mesmo k; o ARI resultante é uma verificação de sensibilidade à escala, não um tratamento universal de outliers ou evidência causal.

In [ ]:
analysis.summary


## Exportação e gráficos
A publicação cria uma geração com CSVs, JSON e gráficos. latest.json só muda quando todos os artefatos são produzidos. Uma falha preserva o ponteiro anterior. Saídas estão ignoradas pelo Git por padrão.

In [ ]:
output = Path(os.environ.get('MARKETING_NOTEBOOK_OUTPUT', str(ROOT / 'outputs/notebook')))
manifest = publish(data, output, synthetic=True)
from IPython.display import Image, display
for name in ['segmentos.png', 'avaliacao.png']:
    display(Image(filename=str(output / manifest['directory'] / name)))
assert manifest['synthetic'] is True
print({'status': 'complete', 'synthetic': True})


## Limites e próximos experimentos
A geometria de dados artificiais não comprova potencial de retenção ou melhora em campanhas. Estabilidade entre sementes não mede estabilidade temporal ou generalização. A avaliação usa as mesmas observações do agrupamento e é interna/exploratória. Em dados autorizados, validar a definição da pontuação, adequação das variáveis, estabilidade por reamostragem/período e hipóteses com experimentos comerciais antes de decisões. Power BI Desktop não é executado por este notebook.